# Ali's architecture pilot — residual feature adapter

Use **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Allow Drive access.
The default data folder matches our previous notebooks. The notebook reuses compatible cached features
and extracts missing features with the project's pinned AraBEST encoder.

We compare three small heads on the same 500 training and 200 development clips.

| Arm | Representation before the same linear classifier |
|---|---|
| linear | Final-layer pooled feature → linear classifier |
| bottleneck | Final-layer feature → 64-unit GELU bottleneck → 848 dimensions → classifier |
| residual | Same bottleneck, with its output added to the input → classifier |

The final-layer feature is L2-normalized in all three arms. This gives a matched
linear reference for this experiment. Historical scores used different feature processing.
The frozen encoder runs once. Each head trains for 100 epochs with seed 11, then evaluates once.
The 200-clip development pilot has already been used in earlier experiments, so this is exploratory screening.

Our hypothesis is that a nonlinear residual transformation can adapt the frozen representation
to dialect classification with few trainable parameters. The bottleneck control measures what
the skip connection adds at the same parameter count. This adapts pooled features.
It is inspired by [bottleneck adapters](https://arxiv.org/abs/1902.00751).
Layer mixing is already covered by Fatima's EEC-29 experiment.

Two further ideas for EEC-28 are low-rank adaptation of AraBEST's attention projections, compared
with frozen and fully fine-tuned encoders, and a head combining frozen AraBEST and Whisper encoder
features, compared with each encoder alone. These need additional implementation and extraction.
References are [LoRA](https://arxiv.org/abs/2106.09685),
[Whisper](https://arxiv.org/abs/2212.04356), and the
[AraBEST model card](https://huggingface.co/Elyadata/AraBEST-RQ-300M-6k).

At the end, the notebook saves checkpoints, predictions, curves, metrics and a summary
on Drive, then downloads a small results ZIP. Send that ZIP back for analysis.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess
import sys
import tempfile
import time

drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/Arabic-Dialect-Identification-data')
AUDIO_ROOT = DATA_ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'
FEATURE_ROOT = DATA_ROOT / 'output'
RUN_ROOT = DATA_ROOT / 'runs/ali_residual_adapter' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_ROOT.mkdir(parents=True, exist_ok=False)
SEED = 11
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.001
BOTTLENECK_DIM = 64
print('Results folder', RUN_ROOT)


In [ ]:
PINNED_COMMIT = '31013e56e52daaa97ce992c06152e1e00d595571'
REPO_ROOT = Path(tempfile.mkdtemp(prefix='adi-residual-adapter-', dir='/content'))
subprocess.run(['git', 'init', '-q', str(REPO_ROOT)], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '-q', '--depth', '1',
                'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git', PINNED_COMMIT], check=True)
subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile==0.14.0'], check=True)
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from src.models import acoustic_encoder as ae
from src.experiments import acoustic_classifier as acoustic
from src.evaluation import evaluate as evaluator

torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
LABELS_PATH = REPO_ROOT / 'configs/labels.json'
LABELS = evaluator.load_labels(LABELS_PATH)
train_manifest = REPO_ROOT / 'data/manifests/train_pilot.jsonl'
dev_manifest = REPO_ROOT / 'data/manifests/dev_pilot.jsonl'
train_metadata = REPO_ROOT / 'data/manifests/train_pilot.metadata.json'
dev_metadata = REPO_ROOT / 'data/manifests/dev_pilot.metadata.json'
train_rows, train_split, train_meta = acoustic.base.load_split(train_manifest, train_metadata, LABELS_PATH, LABELS)
dev_rows, dev_split, dev_meta = acoustic.base.load_split(dev_manifest, dev_metadata, LABELS_PATH, LABELS)
assert train_split == 'train' and dev_split == 'dev'
assert all(train_meta[key] == dev_meta[key] for key in ('dataset_version', 'split_version'))
assert len(train_rows) == 500 and len(dev_rows) == 200
assert not ({r['clip_id'] for r in train_rows} & {r['clip_id'] for r in dev_rows})
print('Training clips', len(train_rows), '| development clips', len(dev_rows))
print('PyTorch', torch.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


### Reuse or extract features

Uses the same all-layer, mean-pooled, full-duration FP32 feature configuration as the earlier pilot.
The encoder processes one clip at a time. Compatible cache entries are reused and every requested ID
is checked. Extraction saves progress in chunks. A failed clip stops the run with its error.


In [ ]:
feature_config = ae.FeatureConfig(layers='all', pooling='mean', precision='fp32', max_duration_sec=None)
cache_path = FEATURE_ROOT / 'features' / ae.ENCODER_NAME / ae.REVISION[:12] / feature_config.sha256()[:16]

def open_cache():
    config_path = cache_path / 'config.json'
    if not config_path.is_file():
        return None
    provenance = json.loads(config_path.read_text())
    expected = json.loads(json.dumps(feature_config.to_dict()))
    assert provenance['feature_config'] == expected, 'Feature settings differ'
    assert provenance['feature_config_sha256'] == feature_config.sha256()
    assert provenance['checkpoint']['files'] == ae.CHECKPOINT_SHA256, 'Checkpoint hashes differ'
    assert provenance['output_dim'] == 848
    assert provenance['layer_names'] == ae.layer_names(ae.Architecture(), range(26))
    return ae.FeatureCache(FEATURE_ROOT, provenance)

cache = open_cache()
all_rows = train_rows + dev_rows
missing = all_rows if cache is None else cache.missing(all_rows)
extraction_start = time.perf_counter()
if missing:
    assert torch.cuda.is_available(), 'Choose a T4 GPU runtime, then run again'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'speechbrain==1.1.2',
                    'huggingface_hub', 'pyarrow', 'PyYAML'], check=True)
    absent_audio = [r for r in missing if not (AUDIO_ROOT / r['audio_path']).is_file()]
    if absent_audio:
        subprocess.run([sys.executable, '-u', str(REPO_ROOT / 'src/data/fetch_pilot_audio.py'),
                        '--audio-root', str(AUDIO_ROOT)], cwd=REPO_ROOT, check=True)
    torch.cuda.reset_peak_memory_stats()
    for manifest in (train_manifest, dev_manifest):
        subprocess.run([sys.executable, '-u', str(REPO_ROOT / 'src/models/acoustic_encoder.py'),
                        '--manifest', str(manifest), '--audio-root', str(AUDIO_ROOT),
                        '--output-root', str(FEATURE_ROOT), '--cache-root', '/content/hf-cache',
                        '--device', 'cuda', '--batch-size', '1', '--chunk-size', '16'],
                       cwd=REPO_ROOT, check=True)
    cache = open_cache()
assert cache is not None and not cache.missing(all_rows), 'Some requested features are missing'
print('Cache', cache.root)
print('Reused at start', len(all_rows) - len(missing), '| extracted', len(missing))
print('Feature preparation seconds', round(time.perf_counter() - extraction_start, 1))


In [ ]:
train_ids = [r['clip_id'] for r in train_rows]
dev_ids = [r['clip_id'] for r in dev_rows]
train_features = cache.load(train_ids)
dev_features = cache.load(dev_ids)
assert train_features.clip_ids == train_ids and dev_features.clip_ids == dev_ids
layer_names = train_features.layer_names
final_index = layer_names.index('final')
train_x = torch.from_numpy(train_features.features).float()
dev_x = torch.from_numpy(dev_features.features).float()
assert tuple(train_x.shape) == (500, 26, 848)
assert tuple(dev_x.shape) == (200, 26, 848)
assert torch.isfinite(train_x).all() and torch.isfinite(dev_x).all()
assert (train_x.norm(dim=-1) > 0).all() and (dev_x.norm(dim=-1) > 0).all()
train_x = F.normalize(train_x[:, final_index], dim=-1)
dev_x = F.normalize(dev_x[:, final_index], dim=-1)
train_y = torch.tensor([LABELS.index(r['label']) for r in train_rows])
assert len(torch.unique(train_y)) == len(LABELS)
print('Feature tensors', tuple(train_x.shape), tuple(dev_x.shape))


In [ ]:
class DialectHead(torch.nn.Module):
    def __init__(self, mode, feature_dim, num_labels, bottleneck_dim, seed):
        super().__init__()
        self.mode = mode
        self.classifier = acoustic.build_head(feature_dim, num_labels, seed)
        if mode != 'linear':
            self.down = acoustic.build_head(feature_dim, bottleneck_dim, seed + 1)
            self.up = acoustic.build_head(bottleneck_dim, feature_dim, seed + 2)

    def forward(self, features):
        if self.mode == 'linear':
            return self.classifier(features)
        adapted = self.up(F.gelu(self.down(features)))
        if self.mode == 'residual':
            adapted = features + adapted
        return self.classifier(adapted)


def train_arm(mode):
    init_seed, shuffle_seed = acoustic.run_seeds(SEED)
    model = DialectHead(mode, train_x.shape[1], len(LABELS), BOTTLENECK_DIM, init_seed)
    groups = [{'params': [p for name, p in model.named_parameters() if not name.endswith('bias')],
               'weight_decay': 0.01},
              {'params': [p for name, p in model.named_parameters() if name.endswith('bias')],
               'weight_decay': 0.0}]
    optimizer = torch.optim.AdamW(groups, lr=LEARNING_RATE)
    generator = torch.Generator().manual_seed(shuffle_seed)
    curve = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        order = torch.randperm(len(train_x), generator=generator)
        for start in range(0, len(order), BATCH_SIZE):
            batch = order[start:start + BATCH_SIZE]
            loss = F.cross_entropy(model(train_x[batch]), train_y[batch])
            assert torch.isfinite(loss), f'Nonfinite loss in {mode}'
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            train_logits = model(train_x)
            row = {'epoch': epoch, 'loss': F.cross_entropy(train_logits, train_y).item(),
                   'accuracy': (train_logits.argmax(1) == train_y).float().mean().item()}
        curve.append(row)
        if epoch == 1 or epoch % 20 == 0:
            print(mode, 'epoch', epoch, '| train loss', round(row['loss'], 4),
                  '| train accuracy', round(row['accuracy'], 4), flush=True)
    return model, curve


In [ ]:
import csv
from importlib.metadata import version, PackageNotFoundError

experiment = {'task': 'EEC-28', 'source_commit': PINNED_COMMIT,
              'seed': SEED, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE,
              'learning_rate': LEARNING_RATE, 'checkpoint_selection': 'last_epoch',
              'optimizer': 'AdamW', 'weight_decay_weights': 0.01, 'weight_decay_bias': 0.0,
              'bottleneck_dim': BOTTLENECK_DIM, 'layer_name': 'final',
              'gradient_clip_norm': 1.0, 'layer_preprocessing': 'L2 normalization of final pooled layer',
              'train_manifest_sha256': evaluator.sha256_file(train_manifest),
              'dev_manifest_sha256': evaluator.sha256_file(dev_manifest),
              'train_metadata': train_meta, 'dev_metadata': dev_meta,
              'feature_provenance': cache.provenance, 'labels': LABELS,
              'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
              'head_device': 'cpu', 'result_role': 'exploratory_500_train_200_dev_pilot'}
experiment['packages'] = {}
for name in ('torch', 'torchaudio', 'speechbrain', 'numpy', 'soundfile', 'huggingface_hub'):
    try:
        experiment['packages'][name] = version(name)
    except PackageNotFoundError:
        experiment['packages'][name] = None
(RUN_ROOT / 'experiment.json').write_text(json.dumps(experiment, indent=2) + '\n')
freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], text=True, capture_output=True, check=True)
(RUN_ROOT / 'environment.txt').write_text(freeze.stdout)
summary = []
for mode in ('linear', 'bottleneck', 'residual'):
    arm_root = RUN_ROOT / mode
    arm_root.mkdir(exist_ok=False)
    started = time.perf_counter()
    model, curve = train_arm(mode)
    training_seconds = time.perf_counter() - started
    checkpoint = arm_root / 'checkpoint.pt'
    torch.save({'state_dict': model.state_dict(), 'mode': mode, 'labels': LABELS,
                'layer_names': layer_names, 'final_index': final_index, 'experiment': experiment}, checkpoint)
    model_revision = 'ali-residual-adapter-' + evaluator.sha256_file(checkpoint)[:16]
    with torch.no_grad():
        scores = model(dev_x).numpy()
    assert scores.shape == (len(dev_rows), len(LABELS)) and np.isfinite(scores).all()
    predictions = [{'clip_id': row['clip_id'], 'predicted_label': LABELS[int(logits.argmax())],
                    'status': 'ok', 'model_revision': model_revision,
                    'scores': logits.tolist(), 'score_labels': LABELS, 'score_type': 'logits'}
                   for row, logits in zip(dev_rows, scores)]
    prediction_path = arm_root / 'predictions.jsonl'
    prediction_path.write_text(''.join(json.dumps(row) + '\n' for row in predictions))
    subprocess.run([sys.executable, str(REPO_ROOT / 'src/evaluation/evaluate.py'),
                    '--manifest', str(dev_manifest), '--metadata', str(dev_metadata),
                    '--predictions', str(prediction_path), '--labels', str(LABELS_PATH),
                    '--split', 'dev', '--model-name', 'ali-residual-adapter-' + mode,
                    '--output-dir', str(arm_root / 'evaluation')], cwd=REPO_ROOT, check=True)
    (arm_root / 'training_curve.json').write_text(json.dumps(curve, indent=2) + '\n')
    metrics = json.loads((arm_root / 'evaluation/metrics.json').read_text())
    summary.append({'arm': mode, 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1'],
                    'coverage': metrics['coverage'], 'train_accuracy': curve[-1]['accuracy'],
                    'train_loss': curve[-1]['loss'], 'training_seconds': training_seconds,
                    'trainable_parameters': sum(p.numel() for p in model.parameters())})
    print('Finished', mode, flush=True)
with (RUN_ROOT / 'summary.csv').open('w', newline='') as stream:
    writer = csv.DictWriter(stream, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
(RUN_ROOT / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
print('\nArm         Dev accuracy    Macro-F1    Coverage')
for row in summary:
    print(f"{row['arm']:<12}{row['accuracy']:>8.2%}       {row['macro_f1']:.4f}      {row['coverage']:.0%}")
print('Results', RUN_ROOT)


Compare residual against bottleneck to measure the effect of the skip connection at the same
parameter count. Compare both against linear to measure the effect of adding the nonlinear
transformation. Check training loss and accuracy alongside development scores. Repeat promising
results on larger data and multiple seeds before selecting a final model.


In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/ali_residual_adapter_' + RUN_ROOT.name, 'zip', RUN_ROOT)
print('Saved on Drive', RUN_ROOT)
files.download(archive)
